In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports and Configs

In [2]:
# ============================================================
# IMPORTS
# ============================================================

import os, gc, time, zipfile, pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.sparse import csr_matrix
from pathlib import Path
import pickle
import shutil

import random

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

# ============================================================
# CONFIGS
# ============================================================
SEED = 42

TRAIN_PATH    = "/content/drive/MyDrive/Colab Notebooks/cs608_ip_train_v3.csv"
PROBE_PATH    = "/content/drive/MyDrive/Colab Notebooks/cs608_ip_probe_v3.csv"

BPR_CACHE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/recsys_cache/trainplusprobe")
BPR_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check — will raise clearly if paths are wrong
import os
assert os.path.exists(TRAIN_PATH), f"Train file not found: {TRAIN_PATH}"
assert os.path.exists(PROBE_PATH), f"Probe file not found: {PROBE_PATH}"
print("Paths OK")
CACHE_DIR     = Path("/content/drive/MyDrive/Colab Notebooks/recsys_cache/trainplusprobe")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

K             = 50
CANDIDATE_K   = 3000
REL_THRESHOLD = 4.0

# Three EASE signal variants — each produces a different item-item weight matrix
EASE_CONFIGS = {
    "ease_all"    : {"col": "implicit_all", "lam": 200,  "min_count": 2},
    "ease_lam200_min1": {"col": "implicit_all", "lam": 200, "min_count": 1},
#     "ease_lam500" : {"col": "implicit_all", "lam": 500,  "min_count": 2},
#     "ease_lam1000": {"col": "implicit_all", "lam": 1000, "min_count": 2},
#     "ease_lam2000": {"col": "implicit_all", "lam": 2000, "min_count": 2},
#     "ease_lam5000": {"col": "implicit_all", "lam": 5000, "min_count": 2},


    # "ease_reweighted": {
    # "col":       "probe_reweighted",
    # "lam":       200,
    # "min_count": 2,
    # },
    # "ease_high": {
    #     "col": "high_conf",          # rating >= 4 -> 1.0, else dropped
    #     "lam": 150, "min_count": 2,
    # },
    # "ease_wt": {
    #     "col": "rating_wt",          # rating / 5.0 as soft confidence weight
    #     "lam": 300, "min_count": 2,
    # }
  }

SUBMISSION_TXT = "submission_v2.txt"
SUBMISSION_ZIP = "submission_v2.zip"

# Leaderboard targets (rank 1 as of 2026-05-02)
# TARGET_HM     = 0.07025
# TARGET_NDCG   = 0.06677
# TARGET_NCRR   = 0.057435
# TARGET_RECALL = 0.09693

# print("Config OK")
# print(f"Cache dir: {CACHE_DIR}")

PyTorch  : 2.10.0+cu128
CUDA     : True | NVIDIA A100-SXM4-40GB
Paths OK


## FUNCTIONS

In [6]:
# ============================================================
# DATA UTILITIES
# ============================================================

def load_data(holdout_frac=0.15, holdout_seed=42):
    """
    Union strategy: pool train + probe into one dataset, then carve out
    our own stratified holdout for local evaluation

    Args:
        holdout_frac : fraction of each user's interactions to hold out
        holdout_seed : RNG seed for reproducibility
    """
    train_raw = pd.read_csv(TRAIN_PATH)
    probe_raw = pd.read_csv(PROBE_PATH)

    for df in [train_raw, probe_raw]:
        df.columns = df.columns.str.strip().str.lower()

    # Union train and probe datasets
    union = pd.concat([train_raw, probe_raw], ignore_index=True)
    union = union.drop_duplicates(subset=["user_id", "item_id"]).reset_index(drop=True)

    print(f"train_raw : {len(train_raw):,}  |  probe_raw : {len(probe_raw):,}")
    print(f"Union (after dedup): {len(union):,}")

    # Build global ID maps from the union
    all_users   = union.user_id.unique()
    all_items   = union.item_id.unique()
    user_map    = {u: i for i, u in enumerate(sorted(all_users))}
    item_map    = {v: i for i, v in enumerate(sorted(all_items))}
    idx_to_item = {i: v for v, i in item_map.items()}

    union["u"] = union.user_id.map(user_map)
    union["v"] = union.item_id.map(item_map)

    n_users = len(user_map)
    n_items = len(item_map)

    # Per-user stratified split
    rng = np.random.default_rng(holdout_seed)
    holdout_mask = np.zeros(len(union), dtype=bool)

    for u_idx, grp in union.groupby("u"):
        n = len(grp)
        if n < 2:
            continue
        n_hold = int(np.floor(n * holdout_frac))

        if n_hold == 0:
            continue

        hold_indices = rng.choice(grp.index.values, size=n_hold, replace=False)
        holdout_mask[hold_indices] = True

    train_df  = union[~holdout_mask].reset_index(drop=True)
    holdout_df = union[holdout_mask].reset_index(drop=True)

    print(f"\nSplit summary (holdout_frac={holdout_frac}):")
    print(f"  Train (new) : {len(train_df):,}  interactions")
    print(f"  Holdout     : {len(holdout_df):,}  interactions")
    print(f"  Users       : {n_users:,}")
    print(f"  Items       : {n_items:,}")
    sparsity = 1 - len(train_df) / (n_users * n_items)
    print(f"  Sparsity    : {sparsity:.6f}")
    print(f"  Avg interactions/user (train): {len(train_df)/train_df.u.nunique():.1f}")
    print(f"\nTrain rating distribution:")
    print(train_df.rating.value_counts().sort_index())
    print(f"\nHoldout users with relevant items (rating>={REL_THRESHOLD}): "
          f"{(holdout_df.rating >= REL_THRESHOLD).groupby(holdout_df.u).any().sum():,}")

    return train_df, holdout_df, n_users, n_items, idx_to_item


def add_signal_columns(df):
    df = df.copy()

    # Original signals
    df["implicit_all"] = 1.0
    df["high_conf"]    = (df["rating"] >= 4).astype("float32")
    df["rating_wt"]    = (df["rating"] / 5.0).astype("float32")

    # User mean-centered: how much more than usual did this user like this item
    user_mean = df.groupby("u")["rating"].transform("mean")
    df["user_mean_centered"] = (df["rating"] - user_mean).clip(lower=0).astype("float32")

    # Item deviation: how much more than average did this user rate this item
    item_mean = df.groupby("v")["rating"].transform("mean")
    df["item_deviation"] = (df["rating"] - item_mean).clip(lower=0).astype("float32")

    # Relative preference: normalised personal preference, bounded 0-1
    df["relative_preference"] = (
        (df["rating"] - user_mean) / (5.0 - user_mean + 1e-10)
    ).clip(lower=0).astype("float32")

    return df

def build_sparse(df, n_users, n_items, col, drop_zeros=True):
    tmp = df[["u", "v", col]].copy().rename(columns={col: "w"})
    tmp["w"] = tmp["w"].astype("float32")
    if drop_zeros:
        tmp = tmp[tmp["w"] > 0]
    return csr_matrix(
        (tmp.w.values, (tmp.u.values, tmp.v.values)),
        shape=(n_users, n_items),
    )


def get_ground_truth(probe_df, threshold=4.0):
    rel = probe_df[probe_df.rating >= threshold]
    return rel.groupby("u")["v"].apply(set).to_dict()


def item_pop(X):  return np.asarray(X.sum(axis=0)).ravel()
def pop_order(X): return np.argsort(-item_pop(X))


def fill_to_k(items, seen, pop_ord, k=50):
    result, used = [], set()
    for v in items:
        if v not in seen and v not in used:
            result.append(v); used.add(v)
        if len(result) == k: return result
    for v in pop_ord:
        if v not in seen and v not in used:
            result.append(v); used.add(v)
        if len(result) == k: return result
    return result


# ============================================================
# METRIC HELPER FUNCTIONS
# ============================================================
def ndcg_at_k(ranked, rel, k):
    dcg  = sum(1/np.log2(r+2) for r, i in enumerate(ranked[:k]) if i in rel)
    idcg = sum(1/np.log2(r+2) for r in range(min(len(rel), k)))
    return dcg/idcg if idcg else 0.0

def ncrr_at_k(ranked, rel, k):
    crr  = sum(1/(r+1) for r, i in enumerate(ranked[:k]) if i in rel)
    icrr = sum(1/(r+1) for r in range(min(len(rel), k)))
    return crr/icrr if icrr else 0.0

def recall_at_k(ranked, rel, k):
    return sum(1 for i in ranked[:k] if i in rel)/len(rel) if rel else 0.0

def evaluate(recs, gt, k=50):
    ns, cs, rs = [], [], []
    for u, rel in gt.items():
        if u not in recs or not rel: continue
        ns.append(ndcg_at_k(recs[u], rel, k))
        cs.append(ncrr_at_k(recs[u], rel, k))
        rs.append(recall_at_k(recs[u], rel, k))
    mn, mc, mr = float(np.mean(ns)), float(np.mean(cs)), float(np.mean(rs))
    hm = 3/(1/mn+1/mc+1/mr) if (mn > 0 and mc > 0 and mr > 0) else 0.0
    return {"ndcg": mn, "ncrr": mc, "recall": mr, "hm": hm, "n": len(ns)}

def fmt(label, res, elapsed=0):
    print(f"\n  ── {label}  ({elapsed:.0f}s)")
    print(f"  NDCG@{K}   : {res['ndcg']:.6f}")
    print(f"  NCRR@{K}   : {res['ncrr']:.6f}")
    print(f"  Recall@{K} : {res['recall']:.6f}")
    print(f"  HM         : {res['hm']:.6f}")

print("Metrics OK")


# ============================================================
# CACHE HELPER FUNTCIONS
# ============================================================
def save_pkl(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_pkl(path):
    p = Path(path)
    if not p.exists(): return None
    with open(p, "rb") as f:
        return pickle.load(f)

print("Cache helpers OK")


# ============================================================
# EASE (Experimental only;)
# ============================================================

def run_ease(X_train, X_seen, probe_users, ground_truth, pop_ord,
             lam=200.0, min_count=2, candidate_k=3000, label="EASE",
             adaptive_lambda=False, G_extra=None):
    t0 = time.time()

    Xcsc        = X_train.tocsc()
    item_counts = np.diff(Xcsc.indptr)
    keep        = np.where(item_counts >= min_count)[0]
    n_keep      = len(keep)
    X_red       = Xcsc[:, keep].toarray().astype("float32")

    print(f"[{label}] items={n_keep:,}  gram={n_keep**2*4/1e9:.2f} GB")

    # Gram matrix on GPU
    X_gpu = torch.tensor(X_red, device="cuda", dtype=torch.float32)
    G_gpu = X_gpu.T @ X_gpu
    del X_gpu; torch.cuda.empty_cache()

    # Optional Experimentation before using data union strat
    # Intention was to use probe data characteristics for signal training
    if G_extra is not None:
        G_extra_keep = G_extra[np.ix_(keep, keep)]
        G_extra_gpu  = torch.tensor(
            G_extra_keep.astype("float32"), device="cuda", dtype=torch.float32
        )
        G_gpu += G_extra_gpu
        del G_extra_gpu; torch.cuda.empty_cache()
        print(f"[{label}] Probe gram augmentation applied  "
              f"nnz={int((G_extra_keep > 0).sum()):,}")

    # Regularization
    if adaptive_lambda:
        item_freq  = torch.tensor(
            item_counts[keep].astype("float32"), device="cuda"
        )
        lambda_vec = lam * (item_freq.mean() / item_freq)
        G_gpu.diagonal().add_(lambda_vec)
        del item_freq, lambda_vec
    else:
        G_gpu.diagonal().add_(lam)

    # Cholesky inversion
    L = torch.linalg.cholesky(G_gpu, upper=False)
    del G_gpu; torch.cuda.empty_cache()

    I = torch.eye(n_keep, device="cuda", dtype=torch.float32)
    Y = torch.linalg.solve_triangular(L, I, upper=False)
    P = torch.linalg.solve_triangular(L.mT, Y, upper=True)
    del L, Y, I; torch.cuda.empty_cache()

    diag_P = P.diagonal().clone()
    W      = P / (-diag_P.unsqueeze(0))
    W.diagonal().fill_(0.0)
    del P; torch.cuda.empty_cache()

    # Chunk inference
    X_gpu = torch.tensor(X_red, device="cuda", dtype=torch.float32)
    CHUNK = 2000
    recs, scores_store = {}, {}

    for start in range(0, len(probe_users), CHUNK):
        batch = probe_users[start:start+CHUNK]
        S_np  = (X_gpu[batch] @ W).cpu().numpy()

        for i, u in enumerate(batch):
            # Mask seen items for this user only
            seen_items = X_seen[u].indices
            local_idx  = np.searchsorted(keep, seen_items)
            in_bounds  = local_idx < n_keep
            matched    = np.zeros_like(in_bounds, dtype=bool)
            matched[in_bounds] = keep[local_idx[in_bounds]] == seen_items[in_bounds]
            S_np[i, local_idx[matched]] = -np.inf

            top_local_i  = np.argsort(-S_np[i])[:candidate_k]
            top_global_i = keep[top_local_i]
            ts           = S_np[i, top_local_i]
            mask         = ts > -np.inf
            vi, vs       = top_global_i[mask], ts[mask]

            seen = set(seen_items)
            scores_store[u] = {int(v): float(s) for v, s in zip(vi, vs)}
            recs[u] = fill_to_k(list(vi), seen, pop_ord, candidate_k)

    del X_gpu, W; torch.cuda.empty_cache(); gc.collect()

    res = evaluate(recs, ground_truth, K)
    fmt(label, res, time.time()-t0)
    return res, recs, scores_store


def run_all_ease_variants(train_df, n_users, n_items, probe_users, gt, pop_ord, X_seen):
    results = {}
    for name, cfg in EASE_CONFIGS.items():
        cache_path = CACHE_DIR / f"{name}_lam{cfg['lam']}_min{cfg['min_count']}_ck{CANDIDATE_K}.pkl"
        cached = load_pkl(cache_path)

        if cached:
            print(f"[CACHE HIT] {name}  HM={cached['res']['hm']:.6f}")
            results[name] = cached
            continue

        X_var = build_sparse(train_df, n_users, n_items, cfg["col"], drop_zeros=True)
        res, recs, scores = run_ease(
            X_var, X_seen, probe_users, gt, pop_ord,
            lam=float(cfg["lam"]), min_count=cfg["min_count"],
            candidate_k=CANDIDATE_K, label=name,
        )
        payload = {"res": res, "recs": recs, "scores": scores}
        save_pkl(payload, cache_path)
        print(f"[CACHE SAVED] {cache_path}")
        results[name] = payload
        del X_var; gc.collect(); torch.cuda.empty_cache()

    return results

print("EASE functions OK")


# ============================================================
# ADMM-SLIM
# ============================================================

# Hyperparams:
#   lambda1: L1 penalty -- controls sparsity (higher = sparser W)
#   lambda2: L2 penalty -- controls magnitude (similar role to EASE lambda)
#   rho:     ADMM step size -- affects convergence speed, not final solution
#   n_iter:  number of ADMM iterations -- 50-100 typically sufficient

def soft_threshold(x, threshold):
    """
    L1 proximal operator -- elementwise soft thresholding
    """
    return np.sign(x) * np.maximum(np.abs(x) - threshold, 0.0)


def run_admm_slim(X_train, X_seen, probe_users, gt, pop_ord,
                  lambda1=1.0, lambda2=500.0, rho=1000.0,
                  n_iter=100, min_count=2, candidate_k=3000, label="ADMM-SLIM"):
    t0 = time.time()

    # Filter items by min_count
    Xcsc        = X_train.tocsc()
    item_counts = np.diff(Xcsc.indptr)
    keep        = np.where(item_counts >= min_count)[0]
    n_keep      = len(keep)
    X_red       = Xcsc[:, keep].toarray().astype("float32")

    print(f"[{label}] items={n_keep:,} lambda1={lambda1} lambda2={lambda2} rho={rho}")

    # Precompute gram matrix + (G + rho*I)^-1 on GPU
    # All succeeding iterations will use this gram matrxi
    X_gpu = torch.tensor(X_red, device="cuda", dtype=torch.float32)
    G_gpu = X_gpu.T @ X_gpu                           # (n_keep, n_keep)
    XtX   = G_gpu.clone()                             # save for later

    # Regularised inverse: P = (G + (lambda2 + rho) * I)^-1
    G_gpu.diagonal().add_(lambda2 + rho)
    L     = torch.linalg.cholesky(G_gpu, upper=False)
    del G_gpu; torch.cuda.empty_cache()

    I_gpu = torch.eye(n_keep, device="cuda", dtype=torch.float32)
    Y     = torch.linalg.solve_triangular(L, I_gpu, upper=False)
    P_gpu = torch.linalg.solve_triangular(L.mT, Y, upper=True)
    del L, Y, I_gpu; torch.cuda.empty_cache()

    # P @ XtX; used in W update every iteration
    PXtX = P_gpu @ XtX                                # (n_keep, n_keep)
    del XtX; torch.cuda.empty_cache()

    print(f"[{label}] Gram inversion done ({time.time()-t0:.0f}s). "
          f"Running {n_iter} ADMM iterations...")

    # ADMM Vars
    # W: item-item weight matrix
    # Z: auxiliary variable (sparse, enforces L1 + non-negativity)
    # U: dual variable (scaled Lagrange multiplier)
    W = np.zeros((n_keep, n_keep), dtype="float32")
    Z = np.zeros((n_keep, n_keep), dtype="float32")
    U = np.zeros((n_keep, n_keep), dtype="float32")

    P_np = P_gpu.cpu().numpy()
    del P_gpu; torch.cuda.empty_cache()

    for it in range(1, n_iter + 1):
        # W update (closed form, GPU)
        # W = P @ (XtX + rho * (Z - U))
        ZU    = torch.tensor(Z - U, device="cuda", dtype=torch.float32)
        W_gpu = torch.tensor(PXtX.cpu().numpy(), device="cuda", dtype=torch.float32) \
                + rho * (torch.tensor(P_np, device="cuda", dtype=torch.float32) @ ZU)
        W     = W_gpu.cpu().numpy()
        del W_gpu, ZU; torch.cuda.empty_cache()

        # Zero diagonal constraint
        np.fill_diagonal(W, 0.0)

        # Z update (soft threshold + non-negativity, CPU)
        # Z = max(soft_threshold(W + U, lambda1/rho), 0)
        Z = np.maximum(soft_threshold(W + U, lambda1 / rho), 0.0)
        np.fill_diagonal(Z, 0.0)

        # U update (dual ascent)
        U = U + W - Z

        if it % 10 == 0:
            residual  = np.abs(W - Z).mean()
            w_nonzero = int((Z != 0).sum())
            w_mean    = float(Z[Z != 0].mean()) if w_nonzero > 0 else 0.0
            print(f"[{label}] iter={it:3d}/{n_iter}  "
                  f"residual={residual:.6f}  "
                  f"nnz={w_nonzero:,}  "
                  f"W_mean={w_mean:.4f}  "
                  f"({time.time()-t0:.0f}s)")

    del PXtX; gc.collect(); torch.cuda.empty_cache()

    # Use Z as final W (sparse, non-negative)
    W_final = Z
    print(f"[{label}] W sparsity: {(W_final == 0).mean()*100:.1f}% zeros  "
          f"nnz={int((W_final != 0).sum()):,}")

    # GPU scoring
    W_gpu = torch.tensor(W_final, device="cuda", dtype=torch.float32)
    X_gpu = torch.tensor(X_red,   device="cuda", dtype=torch.float32)
    del W_final; torch.cuda.empty_cache()

    CHUNK = 2000
    recs, scores_store = {}, {}

    for start in range(0, len(probe_users), CHUNK):
        batch = probe_users[start:start+CHUNK]
        S_np  = (X_gpu[batch] @ W_gpu).cpu().numpy()

        for i, u in enumerate(batch):
            seen_items = X_seen[u].indices
            local_idx  = np.searchsorted(keep, seen_items)
            in_bounds  = local_idx < n_keep
            matched    = np.zeros_like(in_bounds, dtype=bool)
            matched[in_bounds] = keep[local_idx[in_bounds]] == seen_items[in_bounds]
            S_np[i, local_idx[matched]] = -np.inf

        top_local  = np.argsort(-S_np, axis=1)[:, :candidate_k]
        top_global = keep[top_local]

        for i, u in enumerate(batch):
            seen = set(X_seen[u].indices)
            tv, ts = top_global[i], S_np[i, top_local[i]]
            mask = ts > -np.inf
            vi, vs = tv[mask], ts[mask]
            scores_store[u] = {int(v): float(s) for v, s in zip(vi, vs)}
            recs[u] = fill_to_k(list(vi), seen, pop_ord, candidate_k)

    del X_gpu, W_gpu; torch.cuda.empty_cache(); gc.collect()

    res = evaluate(recs, gt, K)
    fmt(label, res, time.time()-t0)
    return res, recs, scores_store


def run_admm_slim_cached(train_df, n_users, n_items, X_seen, probe_users, gt, pop_ord):
    cfg        = ADMM_SLIM_CONFIG
    cache_path = CACHE_DIR / (
        f"admm_slim_l1{cfg['lambda1']}_l2{cfg['lambda2']}"
        f"_rho{cfg['rho']}_iter{cfg['n_iter']}"
        f"_min{cfg['min_count']}_ck{CANDIDATE_K}.pkl"
    )
    cached = load_pkl(cache_path)
    if cached is not None:
        print(f"[CACHE HIT] ADMM-SLIM  HM={cached['res']['hm']:.6f}")
        return cached

    X_var = build_sparse(train_df, n_users, n_items, "implicit_all", drop_zeros=True)
    res, recs, scores = run_admm_slim(
        X_var, X_seen, probe_users, gt, pop_ord,
        lambda1=cfg["lambda1"], lambda2=cfg["lambda2"],
        rho=cfg["rho"], n_iter=cfg["n_iter"],
        min_count=cfg["min_count"], candidate_k=CANDIDATE_K,
    )
    payload = {"res": res, "recs": recs, "scores": scores}
    save_pkl(payload, cache_path)
    print(f"[CACHE SAVED] {cache_path}")
    del X_var; gc.collect(); torch.cuda.empty_cache()
    return payload

print("ADMM-SLIM functions OK")

# ============================================================
# BPRMF
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

class BPRMF(torch.nn.Module):
    def __init__(self, n_users, n_items, factors=256):
        super().__init__()
        self.user_emb = torch.nn.Embedding(n_users, factors)
        self.item_emb = torch.nn.Embedding(n_items, factors)

        torch.nn.init.normal_(self.user_emb.weight, std=0.01)
        torch.nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward(self, users, pos_items, neg_items):
        u = self.user_emb(users)
        i = self.item_emb(pos_items)
        j = self.item_emb(neg_items)

        pos = (u * i).sum(dim=1)
        neg = (u * j).sum(dim=1)

        return pos, neg

def train_bprmf(
    train_df,
    n_users,
    n_items,
    factors=256,
    lr=0.01,
    reg=1e-5,
    epochs=50,
    batch_size=8192,
):
    pos_df = train_df[train_df["rating"] >= 4][["u", "v"]].drop_duplicates()

    user_pos = pos_df.groupby("u")["v"].apply(set).to_dict()

    users_np = pos_df["u"].values.astype(np.int64)
    pos_np = pos_df["v"].values.astype(np.int64)

    model = BPRMF(n_users, n_items, factors=factors).cuda()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=reg)

    n = len(users_np)
    rng = np.random.default_rng(SEED)

    for epoch in range(1, epochs + 1):
        # perm = np.random.permutation(n)
        perm = rng.permutation(n)
        losses = []

        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]

            u = users_np[idx]
            i = pos_np[idx]

            # j = np.random.randint(0, n_items, size=len(idx))
            j = rng.integers(0, n_items, size=len(idx))

            # resample negatives that are actually positives
            for t in range(len(j)):
                while j[t] in user_pos.get(u[t], set()):
                    j[t] = np.random.randint(0, n_items)

            u_t = torch.tensor(u, dtype=torch.long, device="cuda")
            i_t = torch.tensor(i, dtype=torch.long, device="cuda")
            j_t = torch.tensor(j, dtype=torch.long, device="cuda")

            pos, neg = model(u_t, i_t, j_t)
            loss = -F.logsigmoid(pos - neg).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()

            losses.append(loss.item())

        print(f"[BPRMF] epoch={epoch} loss={np.mean(losses):.5f}")

    return model

def eval_bprmf(
    model,
    X_seen,
    probe_users,
    ground_truth,
    pop_ord,
    candidate_k=1000,
):
    model.eval()

    item_factors = model.item_emb.weight.detach()
    recs = {}
    scores_store = {}

    CHUNK = 1000

    with torch.no_grad():
        for start in range(0, len(probe_users), CHUNK):
            batch = probe_users[start:start + CHUNK]

            u_t = torch.tensor(batch, dtype=torch.long, device="cuda")
            U = model.user_emb(u_t)

            S = U @ item_factors.T
            S = S.cpu().numpy()

            for i, u in enumerate(batch):
                seen = set(X_seen[u].indices)

                if seen:
                    S[i, list(seen)] = -np.inf

                top = np.argpartition(-S[i], candidate_k)[:candidate_k]
                top = top[np.argsort(-S[i, top])]

                vals = S[i, top]
                mask = vals > -np.inf

                items = top[mask]
                vals = vals[mask]

                scores_store[u] = {
                    int(v): float(s)
                    for v, s in zip(items, vals)
                }

                recs[u] = fill_to_k(list(items), seen, pop_ord, candidate_k)

    res = evaluate(recs, ground_truth, K)
    fmt("BPRMF", res, 0)

    return res, recs, scores_store


def train_bprmf_min_rating(
    train_df,
    n_users,
    n_items,
    min_rating=3,
    factors=512,
    lr=0.003,
    reg=1e-5,
    epochs=50,
    batch_size=8192,
):
    pos_df = train_df[train_df["rating"] >= min_rating][["u", "v"]].drop_duplicates()
    user_pos = pos_df.groupby("u")["v"].apply(set).to_dict()
    users_np = pos_df["u"].values.astype(np.int64)
    pos_np = pos_df["v"].values.astype(np.int64)

    model = BPRMF(n_users, n_items, factors=factors).cuda()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=reg)

    n = len(users_np)
    rng = np.random.default_rng(SEED)

    for epoch in range(1, epochs + 1):
        perm = rng.permutation(n)
        losses = []

        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            u = users_np[idx]
            i = pos_np[idx]
            j = rng.integers(0, n_items, size=len(idx))

            for t in range(len(j)):
                while j[t] in user_pos.get(u[t], set()):
                    j[t] = rng.integers(0, n_items)

            u_t = torch.tensor(u, dtype=torch.long, device="cuda")
            i_t = torch.tensor(i, dtype=torch.long, device="cuda")
            j_t = torch.tensor(j, dtype=torch.long, device="cuda")

            pos_s, neg_s = model(u_t, i_t, j_t)
            loss = -F.logsigmoid(pos_s - neg_s).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())

        print(f"[BPRMF >= {min_rating}] epoch={epoch} loss={np.mean(losses):.5f}")

    return model

# ============================================================
# SUBMISSION WRITER
# ============================================================
def save_submission(recs, X_seen, idx_to_item, n_users, pop_ord, k=50):
    with open(SUBMISSION_TXT, "w") as f:
        for u in range(n_users):
            seen  = set(X_seen[u].indices)
            items = fill_to_k(recs.get(u, []), seen, pop_ord, k)
            f.write(" ".join(str(idx_to_item[v]) for v in items) + "\n")

    with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(SUBMISSION_TXT)

    with open(SUBMISSION_TXT) as f:
        lines = f.readlines()
    assert len(lines) == n_users and all(len(l.split()) == k for l in lines), \
        "Submission format check failed"
    print(f"\n✓ {SUBMISSION_ZIP}  ({os.path.getsize(SUBMISSION_ZIP)/1e3:.0f} KB)")
    print(f"  {len(lines):,} users × {k} items")
    print(f"  First: {lines[0][:70].strip()}")
    print(f"  Last : {lines[-1][:70].strip()}")

print("Submission writer OK")

Metrics OK
Cache helpers OK
EASE functions OK
ADMM-SLIM functions OK
Submission writer OK


## MAIN

In [4]:
# ============================================================
# LOAD DATA
# ============================================================
# Union strategy: train on ALL data; local testing and exploration
# done with 15% startified holdout

train_df, holdout_df, n_users, n_items, idx_to_item = load_data(
    holdout_frac=0.15,   # hold out 15% of each user's interactions for local eval
    holdout_seed=42,
)

train_df = add_signal_columns(train_df)

# X_seen; what each user has already interacted with (used for masking in inference)
X_seen = build_sparse(train_df, n_users, n_items, "implicit_all")

# Ground truth comes from the holdout, not the original probe file
gt          = get_ground_truth(holdout_df, REL_THRESHOLD)
probe_users = np.array(holdout_df.u.unique())

# Popularity stats derived from training data only
i_pop   = item_pop(X_seen)
pop_ord = pop_order(X_seen)

train_raw : 165,008  |  probe_raw : 73,943
Union (after dedup): 238,951

Split summary (holdout_frac=0.15):
  Train (new) : 211,966  interactions
  Holdout     : 26,985  interactions
  Users       : 18,937
  Items       : 51,173
  Sparsity    : 0.999781
  Avg interactions/user (train): 11.2

Train rating distribution:
rating
1      8020
2      5525
3     12370
4     30955
5    155096
Name: count, dtype: int64

Holdout users with relevant items (rating>=4.0): 17,143


In [ ]:
# # ============================================================
# # RUN EASE VARIANTS (EXPLORATORY; MODEL NOT USED IN FINAL SUBMISSION)
# # ============================================================
# print("=" * 60)
# print("EASE LAMBDA SWEEP")
# print("=" * 60)

# ease_results = run_all_ease_variants(
#     train_df, n_users, n_items, probe_users, gt, pop_ord, X_seen
# )

# print("\nEASE lambda sweep summary:")
# best_name, best_hm = None, -1
# for name, payload in ease_results.items():
#     r = payload["res"]
#     marker = " ◄ best" if r["hm"] > best_hm else ""
#     if r["hm"] > best_hm:
#         best_hm, best_name = r["hm"], name
#     print(f"  {name:<15} HM={r['hm']:.6f}  NDCG={r['ndcg']:.6f}  "
#           f"NCRR={r['ncrr']:.6f}  Recall={r['recall']:.6f}{marker}")

# print(f"\nBest: {best_name}  HM={best_hm:.6f}")
# print(f"Projected leaderboard: {best_hm * 1.56:.6f}")

EASE LAMBDA SWEEP
[CACHE HIT] ease_all  HM=0.040509
[CACHE HIT] ease_lam200_min1  HM=0.040700

EASE lambda sweep summary:
  ease_all        HM=0.040509  NDCG=0.038884  NCRR=0.027903  Recall=0.079995 ◄ best
  ease_lam200_min1 HM=0.040700  NDCG=0.039076  NCRR=0.028003  Recall=0.080589 ◄ best

Best: ease_lam200_min1  HM=0.040700
Projected leaderboard: 0.063492


In [ ]:
# ============================================================
# RUN ADMM-SLIM (MIN COUNT 2)
# ============================================================
ADMM_SWEEP = [
    {"lambda1": 0.10, "lambda2": 100,  "rho": 40, "n_iter": 20, "min_count": 2},
    {"lambda1": 0.10, "lambda2": 500,  "rho": 40, "n_iter": 20, "min_count": 2},
    {"lambda1": 0.50, "lambda2": 200,  "rho": 40, "n_iter": 20, "min_count": 2},
    {"lambda1": 0.10, "lambda2": 50,   "rho": 40, "n_iter": 20, "min_count": 2},
    {"lambda1": 0.10, "lambda2": 1000, "rho": 40, "n_iter": 20, "min_count": 2},
]

admm_sweep_results = {}
best_admm_hm, best_admm_cfg, best_admm_result = -1, None, None

print("=" * 60)
print("ADMM-SLIM LAMBDA SWEEP")
print("=" * 60)

for cfg in ADMM_SWEEP:
    ADMM_SLIM_CONFIG = cfg
    label = (f"l1={cfg['lambda1']} l2={cfg['lambda2']} "
             f"rho={cfg['rho']} iter={cfg['n_iter']}")
    print(f"\n── {label} ──")

    result = run_admm_slim_cached(
        train_df, n_users, n_items, X_seen, probe_users, gt, pop_ord
    )

    hm     = result["res"]["hm"]
    marker = " <<< best" if hm > best_admm_hm else ""
    if hm > best_admm_hm:
        best_admm_hm, best_admm_cfg, best_admm_result = hm, label, result

    admm_sweep_results[label] = result
    print(f"  HM={hm:.6f}  NDCG={result['res']['ndcg']:.6f}  "
          f"NCRR={result['res']['ncrr']:.6f}  "
          f"Recall={result['res']['recall']:.6f}  "
          f"projected LB={hm*1.56:.6f}{marker}")

ADMM-SLIM LAMBDA SWEEP

── l1=0.1 l2=100 rho=40 iter=20 ──
[CACHE HIT] ADMM-SLIM  HM=0.042140
  HM=0.042140  NDCG=0.039824  NCRR=0.030702  Recall=0.074025  projected LB=0.065739 <<< best

── l1=0.1 l2=500 rho=40 iter=20 ──
[CACHE HIT] ADMM-SLIM  HM=0.041351
  HM=0.041351  NDCG=0.039498  NCRR=0.028822  Recall=0.079774  projected LB=0.064508

── l1=0.5 l2=200 rho=40 iter=20 ──
[CACHE HIT] ADMM-SLIM  HM=0.042172
  HM=0.042172  NDCG=0.039979  NCRR=0.030261  Recall=0.076458  projected LB=0.065788 <<< best

── l1=0.1 l2=50 rho=40 iter=20 ──
[CACHE HIT] ADMM-SLIM  HM=0.040653
  HM=0.040653  NDCG=0.038271  NCRR=0.030463  Recall=0.067390  projected LB=0.063419

── l1=0.1 l2=1000 rho=40 iter=20 ──
[CACHE HIT] ADMM-SLIM  HM=0.040864
  HM=0.040864  NDCG=0.039171  NCRR=0.028123  Recall=0.081127  projected LB=0.063749


In [ ]:
# ============================================================
# RUN ADMM-SLIM MIN COUNT 1
# ============================================================

# ADMM_SLIM_CONFIG = {"lambda1": 0.1, "lambda2": 200, "rho": 40, "n_iter": 20, "min_count": 1}
ADMM_SLIM_CONFIG = {"lambda1": 0.5, "lambda2": 200, "rho": 40, "n_iter": 20, "min_count": 1}
label = f"l1={ADMM_SLIM_CONFIG['lambda1']} l2={ADMM_SLIM_CONFIG['lambda2']} min_count=1"

print("=" * 60)
print(f"ADMM-SLIM min_count=1 benchmark")
print("=" * 60)
print(f"\n── {label} ──")

admm_min1_result = run_admm_slim_cached(
    train_df, n_users, n_items, X_seen, probe_users, gt, pop_ord
)

r = admm_min1_result["res"]
print(f"\n  HM={r['hm']:.6f}  NDCG={r['ndcg']:.6f}  "
      f"NCRR={r['ncrr']:.6f}  Recall={r['recall']:.6f}  "
      f"projected LB={r['hm']*1.56:.6f}")
print(f"\nvs ADMM min_count=2: HM=0.041413  Recall=0.078090")
print(f"vs EASE baseline:    HM=0.040509  Recall=0.079995")

ADMM-SLIM min_count=1 benchmark

── l1=0.1 l2=200 min_count=1 ──
[ADMM-SLIM] items=49,433 lambda1=0.1 lambda2=200 rho=40
[ADMM-SLIM] Gram inversion done (38s). Running 20 ADMM iterations...
[ADMM-SLIM] iter= 10/20  residual=0.000001  nnz=2,613,762  W_mean=0.0036  (723s)
[ADMM-SLIM] iter= 20/20  residual=0.000000  nnz=2,613,682  W_mean=0.0036  (1402s)
[ADMM-SLIM] W sparsity: 99.9% zeros  nnz=2,613,682

  ── ADMM-SLIM  (1498s)
  NDCG@50   : 0.040131   (target 0.06677)
  NCRR@50   : 0.030235   (target 0.057435)
  Recall@50 : 0.077146   (target 0.09693)
  HM         : 0.042280   (target 0.07025)
[CACHE SAVED] /content/drive/MyDrive/Colab Notebooks/recsys_cache/trainplusprobe/admm_slim_l10.1_l2200_rho40_iter20_min1_ck3000.pkl

  HM=0.042280  NDCG=0.040131  NCRR=0.030235  Recall=0.077146  projected LB=0.065957

vs ADMM min_count=2: HM=0.041413  Recall=0.078090
vs EASE baseline:    HM=0.040509  Recall=0.079995


In [ ]:
# ============================================================
# CELL — BPR HYPERPARAMETER SWEEP (85% union, local eval)
# ============================================================

BPR_SWEEP = [
    {"min_rating": 3, "factors": 512, "lr": 0.003, "reg": 1e-5, "epochs": 100, "batch_size": 8192},
    {"min_rating": 4, "factors": 512, "lr": 0.003, "reg": 1e-5, "epochs": 100, "batch_size": 8192},
    {"min_rating": 4, "factors": 512, "lr": 0.003, "reg": 1e-5, "epochs": 200, "batch_size": 8192},
    {"min_rating": 4, "factors": 512, "lr": 0.003, "reg": 1e-5, "epochs": 300, "batch_size": 8192},
    {"min_rating": 4, "factors": 512, "lr": 0.003, "reg": 1e-5, "epochs": 400, "batch_size": 8192},
    {"min_rating": 4, "factors": 512, "lr": 0.001, "reg": 1e-5, "epochs": 300, "batch_size": 8192},
]

best_bpr_hm, best_bpr_cfg, best_bpr_result = -1, None, None

print("=" * 60)
print("BPR HYPERPARAMETER SWEEP")
print("=" * 60)

for params in BPR_SWEEP:
    label = (f"r{params['min_rating']}_f{params['factors']}"
             f"_lr{params['lr']}_ep{params['epochs']}")

    cache_name = (
        f"BPRMF_r{params['min_rating']}"
        f"_f{params['factors']}"
        f"_lr{params['lr']}"
        f"_reg{params['reg']}"
        f"_ep{params['epochs']}"
        f"_bs{params['batch_size']}"
        f"_ck{CANDIDATE_K}"
        f"_seed{SEED}.pkl"
    )
    cache_path = BPR_CACHE_DIR / cache_name
    cached = load_pkl(cache_path)

    if cached is not None:
        print(f"[CACHE HIT] {label}")
        result = cached
    else:
        print(f"[CACHE MISS] Training {label}...")
        set_seed(SEED)

        model = train_bprmf_min_rating(
            train_df=train_df,
            n_users=n_users,
            n_items=n_items,
            min_rating=params["min_rating"],
            factors=params["factors"],
            lr=params["lr"],
            reg=params["reg"],
            epochs=params["epochs"],
            batch_size=params["batch_size"],
        )

        res, recs, scores = eval_bprmf(
            model=model,
            X_seen=X_seen,
            probe_users=probe_users,
            ground_truth=gt,
            pop_ord=pop_ord,
            candidate_k=CANDIDATE_K,
        )

        result = {
            "label":  label,
            "params": params,
            "res":    res,
            "recs":   recs,
            "scores": scores,
        }
        save_pkl(result, cache_path)
        print(f"[CACHE SAVED] {cache_path}")
        del model; gc.collect(); torch.cuda.empty_cache()

    hm     = result["res"]["hm"]
    marker = " <<< best" if hm > best_bpr_hm else ""
    if hm > best_bpr_hm:
        best_bpr_hm, best_bpr_cfg, best_bpr_result = hm, label, result

    print(f"  {label:<45} HM={hm:.6f}  "
          f"NDCG={result['res']['ndcg']:.6f}  "
          f"NCRR={result['res']['ncrr']:.6f}  "
          f"Recall={result['res']['recall']:.6f}"
          f"  projected LB={hm*1.56:.6f}{marker}")

print(f"\nBest BPR config : {best_bpr_cfg}  HM={best_bpr_hm:.6f}")

BPR HYPERPARAMETER SWEEP
[CACHE HIT] r3_f512_lr0.003_ep100
  r3_f512_lr0.003_ep100                         HM=0.021504  NDCG=0.023375  NCRR=0.012233  Recall=0.066726  projected LB=0.033546 <<< best
[CACHE HIT] r4_f512_lr0.003_ep100
  r4_f512_lr0.003_ep100                         HM=0.021278  NDCG=0.023455  NCRR=0.011945  Recall=0.068308  projected LB=0.033193
[CACHE HIT] r4_f512_lr0.003_ep200
  r4_f512_lr0.003_ep200                         HM=0.021533  NDCG=0.023616  NCRR=0.012133  Recall=0.068702  projected LB=0.033592 <<< best
[CACHE HIT] r4_f512_lr0.003_ep300
  r4_f512_lr0.003_ep300                         HM=0.021201  NDCG=0.023483  NCRR=0.011846  Recall=0.068944  projected LB=0.033073
[CACHE HIT] r4_f512_lr0.003_ep400
  r4_f512_lr0.003_ep400                         HM=0.021288  NDCG=0.023549  NCRR=0.011905  Recall=0.069135  projected LB=0.033209
[CACHE HIT] r4_f512_lr0.001_ep300
  r4_f512_lr0.001_ep300                         HM=0.021812  NDCG=0.023980  NCRR=0.012273  Recall=0.069

In [ ]:
# ============================================================
# Unique recall diagnostic: BPR vs ADMM-SLIM
# ============================================================
# Quantifies how many relevant items BPR retrieves that ADMM-SLIM misses entirely.
# This motivates the two-stage candidate union pipeline further down

# Load 85% ADMM l1=0.50
admm_85 = load_pkl(CACHE_DIR / "admm_slim_l10.5_l2200_rho40_iter20_min2_ck3000.pkl")
admm_scores = admm_85["scores"]

# Load 85% BPR r4
bpr_85 = load_pkl(CACHE_DIR / "BPRMF_r4_f512_lr0.003_reg1e-05_ep100_bs8192_ck3000_seed42.pkl")
bpr_scores = bpr_85["scores"]

# Diagnostic
bpr_unique_hits = []
for u in probe_users:
    admm_items_u = set(admm_scores.get(u, {}).keys())
    bpr_items_u  = set(bpr_scores.get(u, {}).keys())
    relevant     = gt.get(u, set())
    if not relevant:
        continue
    bpr_only = bpr_items_u - admm_items_u
    hits     = len(bpr_only & relevant)
    bpr_unique_hits.append(hits / len(relevant))

print(f"BPR unique recall contribution : {np.mean(bpr_unique_hits):.4f}")
print(f"i.e. {np.mean(bpr_unique_hits)*100:.1f}% of relevant items only BPR finds")

# Item-level disjointness
all_bpr_items  = set().union(*[set(bpr_scores.get(u, {}).keys()) for u in probe_users])
all_admm_items = set().union(*[set(admm_scores.get(u, {}).keys()) for u in probe_users])
bpr_only_items = all_bpr_items - all_admm_items
print(f"\nTotal BPR items considered  : {len(all_bpr_items):,}")
print(f"BPR-unique items (not in ADMM): {len(bpr_only_items):,}")

BPR unique recall contribution : 0.2047
i.e. 20.5% of relevant items only BPR finds

Total BPR items considered  : 41,402
BPR-unique items (not in ADMM): 10,524


### FULL UNION TRAIN

In [ ]:
# Reload full union data
print("Reloading data with holdout_frac=0.0 (full union)...")
train_full, holdout_full, n_users_f, n_items_f, idx_to_item_f = load_data(
    holdout_frac=0.0,
    holdout_seed=42,
)

assert len(holdout_full) == 0, f"Expected 0 holdout got {len(holdout_full):,}"
assert len(train_full) == 238951, f"Expected 238,951 got {len(train_full):,}"
print(f"Full union verified: {len(train_full):,} interactions")

train_full  = add_signal_columns(train_full)
X_seen_full = build_sparse(train_full, n_users_f, n_items_f, "implicit_all")
i_pop_f     = item_pop(X_seen_full)
pop_ord_f   = pop_order(X_seen_full)
all_users_f = np.arange(n_users_f)

In [ ]:
# ============================================================
# BPR FULL UNION TRAIN
# ============================================================
BPR_FULL_PARAMS = {
    "min_rating": 4,
    "factors":    512,
    "lr":         0.001,
    "reg":        1e-5,
    "epochs":     300,
    "batch_size": 8192,
}

bpr_full_cache_name = (
    f"BPRMF_FULLUNION"
    f"_r{BPR_FULL_PARAMS['min_rating']}"
    f"_f{BPR_FULL_PARAMS['factors']}"
    f"_lr{BPR_FULL_PARAMS['lr']}"
    f"_reg{BPR_FULL_PARAMS['reg']}"
    f"_ep{BPR_FULL_PARAMS['epochs']}"
    f"_bs{BPR_FULL_PARAMS['batch_size']}"
    f"_ck{CANDIDATE_K}"
    f"_seed{SEED}.pkl"
)
bpr_full_cache_path = CACHE_DIR / bpr_full_cache_name

bpr_full_cached = load_pkl(bpr_full_cache_path)

if bpr_full_cached is not None:
    print(f"[CACHE HIT] BPR full union loaded from {bpr_full_cache_path}")
    bpr_full_scores = bpr_full_cached["scores"]
else:
    print("[CACHE MISS] Training BPR on full union...")
    set_seed(SEED)

    model = train_bprmf_min_rating(
        train_df=train_full,
        n_users=n_users_f,
        n_items=n_items_f,
        min_rating=BPR_FULL_PARAMS["min_rating"],
        factors=BPR_FULL_PARAMS["factors"],
        lr=BPR_FULL_PARAMS["lr"],
        reg=BPR_FULL_PARAMS["reg"],
        epochs=BPR_FULL_PARAMS["epochs"],
        batch_size=BPR_FULL_PARAMS["batch_size"],
    )

    _, recs_bpr_full, bpr_full_scores = eval_bprmf(
        model=model,
        X_seen=X_seen_full,
        probe_users=all_users_f,
        ground_truth={},
        pop_ord=pop_ord_f,
        candidate_k=CANDIDATE_K,
    )

    payload = {
        "recs":         recs_bpr_full,
        "scores":       bpr_full_scores,
        "params":       BPR_FULL_PARAMS,
        "n_users":      n_users_f,
        "n_items":      n_items_f,
        "train_size":   len(train_full),
        "holdout_frac": 0.0,
    }
    save_pkl(payload, bpr_full_cache_path)
    print(f"[CACHE SAVED] {bpr_full_cache_path}")

    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ============================================================
# ADMM min_count=1 + BPR Full Union Two-Stage Submission
# ============================================================

# All time best is ADMM SLIM MIN1 + BPR4 300 epoch Boost 0.29 LR 0.001 @ 0.074641

set_seed(SEED)

# Train ADMM-SLIM l1=0.50 min_count=1 on full union
ADMM_MIN1_CONFIG = {
    "lambda1":   0.5,
    "lambda2":   200.0,
    "rho":       40.0,
    "n_iter":    20,
    "min_count": 1,
}

admm_min1_cache = CACHE_DIR / (
    f"admm_slim_FULLUNION"
    f"_l1{ADMM_MIN1_CONFIG['lambda1']}"
    f"_l2{ADMM_MIN1_CONFIG['lambda2']}"
    f"_rho{ADMM_MIN1_CONFIG['rho']}"
    f"_iter{ADMM_MIN1_CONFIG['n_iter']}"
    f"_min{ADMM_MIN1_CONFIG['min_count']}"
    f"_ck{CANDIDATE_K}.pkl"
)

cached_admm = load_pkl(admm_min1_cache)

if cached_admm is not None:
    print(f"[CACHE HIT] ADMM min_count=1 full union")
    admm_min1_scores = cached_admm["scores"]
else:
    print("[CACHE MISS] Training ADMM-SLIM l1=0.50 min_count=1 on full union...")
    X_full = build_sparse(train_full, n_users_f, n_items_f, "implicit_all", drop_zeros=True)

    res_admm, recs_admm, scores_admm = run_admm_slim(
        X_full, X_seen_full, all_users_f,
        {}, pop_ord_f,
        lambda1=ADMM_MIN1_CONFIG["lambda1"],
        lambda2=ADMM_MIN1_CONFIG["lambda2"],
        rho=ADMM_MIN1_CONFIG["rho"],
        n_iter=ADMM_MIN1_CONFIG["n_iter"],
        min_count=ADMM_MIN1_CONFIG["min_count"],
        candidate_k=CANDIDATE_K,
    )

    payload = {
        "res":          res_admm,
        "recs":         recs_admm,
        "scores":       scores_admm,
        "config":       ADMM_MIN1_CONFIG,
        "n_users":      n_users_f,
        "n_items":      n_items_f,
        "train_size":   len(train_full),
        "holdout_frac": 0.0,
    }
    save_pkl(payload, admm_min1_cache)
    print(f"[CACHE SAVED] {admm_min1_cache}")

    admm_min1_scores = scores_admm
    del X_full; gc.collect(); torch.cuda.empty_cache()

# Load BPR ep300 full union
BPR_FULL_PARAMS = {
    "min_rating": 4, # Min 4 is best
    "factors":    512,
    "lr":         0.001, # More stable, was 0.003
    "reg":        1e-5,
    "epochs":     300, # Plateus ~250+
    "batch_size": 8192,
}

bpr_full_cache_name = (
    f"BPRMF_FULLUNION"
    f"_r{BPR_FULL_PARAMS['min_rating']}"
    f"_f{BPR_FULL_PARAMS['factors']}"
    f"_lr{BPR_FULL_PARAMS['lr']}"
    f"_reg{BPR_FULL_PARAMS['reg']}"
    f"_ep{BPR_FULL_PARAMS['epochs']}"
    f"_bs{BPR_FULL_PARAMS['batch_size']}"
    f"_ck{CANDIDATE_K}"
    f"_seed{SEED}.pkl"
)

bpr_full_cached = load_pkl(CACHE_DIR / bpr_full_cache_name)
if bpr_full_cached is None:
    raise RuntimeError("BPR ep300 full union cache missing — train BPR first")
bpr_full_scores = bpr_full_cached["scores"]
print(f"[CACHE HIT] BPR ep300 full union: {len(bpr_full_scores):,} users")

# Two-stage candidate union
def two_stage_candidate_union_submission(
    base_scores, aux_scores, X_seen, users, pop_ord,
    aux_boost=0.25, k=50,
):
    final = {}
    for u in users:
        seen = set(X_seen[u].indices)
        base = base_scores.get(u, {})
        aux  = aux_scores.get(u, {})

        def norm(d):
            if not d: return {}
            vals = np.array(list(d.values()), dtype=np.float32)
            lo, hi = vals.min(), vals.max()
            if hi == lo: return {item: 1.0 for item in d}
            return {item: float((v-lo)/(hi-lo)) for item, v in d.items()}

        base_n = norm(base)
        aux_n  = norm(aux)

        all_candidates = set(base_n.keys()) | set(aux_n.keys())
        scored = {}
        for item in all_candidates:
            if item in seen: continue
            if item in base_n:
                scored[item] = base_n[item] + aux_boost * aux_n.get(item, 0.0)
            else:
                scored[item] = aux_boost * aux_n[item]

        ranked = sorted(scored, key=scored.get, reverse=True)
        final[u] = fill_to_k(ranked, seen, pop_ord, k)

    return final


print(f"\nADMM min1 full union users : {len(admm_min1_scores):,}")
print(f"BPR ep300 full union users : {len(bpr_full_scores):,}")

# Generate submissions
for boost in [0.29]: # 0.29 is best
    print(f"\nRunning two-stage (boost={boost})...")
    recs = two_stage_candidate_union_submission(
        base_scores=admm_min1_scores,
        aux_scores=bpr_full_scores,
        X_seen=X_seen_full,
        users=all_users_f,
        pop_ord=pop_ord_f,
        aux_boost=boost,
    )

    label           = f"2stage_admmMIN1_l050_bpr300_b{boost}"
    submission_path = Path(SUBMISSION_ZIP).parent / f"submission_{label}.zip"

    save_submission(recs, X_seen_full, idx_to_item_f, n_users_f, pop_ord_f)
    shutil.copy(SUBMISSION_ZIP, submission_path)
    print(f"[SUBMISSION] {submission_path}")

    from google.colab import files
    files.download(str(submission_path))

print("\n Submissions generated.")